In [ ]:
import pandas as pd
import re

# ==================== 配置区 ====================
# 所有需要处理的 CSV 文件路径（可以写多个）
INPUT_FILES = [
    "extracted_lexicon.csv",
    # "sensitive_words_2.csv",
    # 在这里继续添加你的文件路径 ...
]

OUTPUT_FILE = "sensitive_words_cleaned.csv"   # 最终输出的文件名
KEEP = "first"                                # 去重策略：first 保留第一条记录

# 分隔符替换规则：这些字符只要出现在 word 字段中，一律替换为 &
# （你可以根据实际数据情况自定义这个列表）
SEPARATOR_PATTERN = re.compile(r'\s*[,;，；/|、]\s*|\s+')
# 解释：
#   \s*[,;，；/|、]\s*  → 匹配逗号、分号（中英文）、斜杠、竖线、顿号，周围可能带空格
#   \s{2,}             → 匹配 2 个及以上连续空格
# 替换后还会把首尾空白清掉
# ===============================================

def normalize_word(word: str) -> str:
    """将所有分隔符（包括空格）统一替换为 &，并清理多余 &"""
    if pd.isna(word):
        return ""
    word = str(word)
    # 1. 所有分隔符 -> &
    normalized = SEPARATOR_PATTERN.sub("&", word)
    # 2. 合并连续 &
    normalized = re.sub(r"&{2,}", "&", normalized)
    # 3. 去掉首尾多余的 & 和空格
    normalized = normalized.strip("& ")
    return normalized

def main():
    # 1. 读取所有文件并合并
    all_dfs = []
    for fpath in INPUT_FILES:
        try:
            df = pd.read_csv(fpath, encoding="utf-8")
            print(f"✓ 读取 {fpath}，形状 {df.shape}")
            all_dfs.append(df)
        except UnicodeDecodeError:
            df = pd.read_csv(fpath, encoding="latin-1")
            print(f"✓ 读取 {fpath} (latin-1)，形状 {df.shape}")
            all_dfs.append(df)
        except Exception as e:
            print(f"✗ 读取 {fpath} 失败: {e}")
            continue

    if not all_dfs:
        print("没有成功读取任何文件，退出。")
        return

    full_df = pd.concat(all_dfs, ignore_index=True)
    print(f"\n合并后总记录数: {len(full_df)}")

    # 确保必需的列存在
    required_cols = ["word", "category", "meaning"]
    for col in required_cols:
        if col not in full_df.columns:
            raise KeyError(f"缺失必要列 '{col}'，现有列: {full_df.columns.tolist()}")

    # 2. 规范化 word 列
    print("正在规范化分隔符 ...")
    full_df["word"] = full_df["word"].apply(normalize_word)
    # 过滤掉空的 word
    full_df = full_df[full_df["word"] != ""]

    # 3. 去重（按 word 唯一，保留第一条出现的 category 和 meaning）
    before = len(full_df)
    dedup_df = full_df.drop_duplicates(subset=["word"], keep=KEEP)
    after = len(dedup_df)
    print(f"去重完成: {before} → {after} (移除 {before - after} 条)")

    # 4. 按 category 分组并排序（可选，方便查看）
    dedup_df = dedup_df.sort_values(by=["category", "word"]).reset_index(drop=True)

    # 5. 保存结果
    dedup_df[["word", "category", "meaning"]].to_csv(
        OUTPUT_FILE, index=False, encoding="utf-8"
    )
    print(f"\n最终结果已保存至: {OUTPUT_FILE}")
    print(f"共 {len(dedup_df)} 个唯一敏感词，类别分布如下：")
    print(dedup_df["category"].value_counts().to_string())

if __name__ == "__main__":
    main()

✓ 读取 extracted_lexicon.csv，形状 (13456, 4)

合并后总记录数: 13456
正在规范化分隔符 ...
去重完成: 13456 → 9036 (移除 4420 条)

最终结果已保存至: sensitive_words_cleaned.csv
共 9036 个唯一敏感词，类别分布如下：
category
涉敏      2094
低俗色情    2001
宗教      1675
涉政      1267
暴力恐怖     940
价值观      802
违法       239
未成年       16
国家安全       1
涉宗教        1
